In [1]:
import requests
import pandas as pd
import json
import time
import re

API_BASE = "https://hansard-api.parliament.uk"

HEADERS = {
    "User-Agent": (
        "KU-Leuven-semantic-change-research/1.0 "
        "(academic Master's thesis; "
        "contact: isaac.muscat@student.kuleuven.be)"
    ),
    "Accept": "application/json"
}

session = requests.Session()
session.headers.update(HEADERS)

TARGET_WORDS = [
    "war",
    "peace",
    "security"
]

In [2]:
url = f"{API_BASE}/overview/firstyear.json"

response = session.get(
    url,
    timeout=30
)

print("Status:", response.status_code)
print("URL:", response.url)
print("Content type:", response.headers.get("Content-Type"))

print("\nResponse:")
print(response.text[:2000])

response.raise_for_status()

Status: 200
URL: https://hansard-api.parliament.uk/overview/firstyear.json
Content type: application/json

Response:
1803


In [3]:
TEST_DATE = "1949-05-02"
TEST_HOUSE = "Commons"

url = (
    f"{API_BASE}/historicsittingdays/"
    f"{TEST_HOUSE}/{TEST_DATE}"
)

params = {
    "flattenStructure": True
}

response = session.get(
    url,
    params=params,
    timeout=30
)

print("Status:", response.status_code)
print("URL:", response.url)

if response.status_code != 200:
    print("\nResponse:")
    print(response.text[:5000])

response.raise_for_status()

historic_day = response.json()

print(
    json.dumps(
        historic_day,
        indent=2
    )[:15000]
)

Status: 200
URL: https://hansard-api.parliament.uk/historicsittingdays/Commons/1949-05-02?flattenStructure=True
{
  "Item": {
    "SittingDate": "1949-05-02T00:00:00",
    "House": "Commons",
    "SittingSections": [
      {
        "Name": "Commons Chamber",
        "ExternalId": "be3f5896-412f-4e4b-8aa0-b2a0f7a3c320",
        "DebateSections": [
          {
            "Title": "Commons Chamber",
            "ExternalId": "be3f5896-412f-4e4b-8aa0-b2a0f7a3c320",
            "ParentExternalId": null
          },
          {
            "Title": "Commons Chamber > House Of Commons",
            "ExternalId": "185d6cde-14fa-4815-9eeb-dbb91f24f3b0",
            "ParentExternalId": "be3f5896-412f-4e4b-8aa0-b2a0f7a3c320"
          },
          {
            "Title": "Commons Chamber > Prayers",
            "ExternalId": "45c2bf8a-915c-42f1-acde-0410ed6f426d",
            "ParentExternalId": "be3f5896-412f-4e4b-8aa0-b2a0f7a3c320"
          },
          {
            "Title": "Commons Chamber

In [4]:
def inspect_json(
    obj,
    depth=0,
    max_depth=5
):
    indent = "    " * depth

    if depth > max_depth:
        return

    if isinstance(obj, dict):

        for key, value in obj.items():

            print(
                f"{indent}{key}: "
                f"{type(value).__name__}"
            )

            if isinstance(
                value,
                (dict, list)
            ):
                inspect_json(
                    value,
                    depth + 1,
                    max_depth
                )

    elif isinstance(obj, list):

        print(
            f"{indent}[{len(obj)} items]"
        )

        if obj:
            inspect_json(
                obj[0],
                depth + 1,
                max_depth
            )


if "historic_day" in globals():
    inspect_json(historic_day)
else:
    print(
        "historic_day has not been loaded yet. "
        "Run the previous cell successfully first."
    )

Item: dict
    SittingDate: str
    House: str
    SittingSections: list
        [2 items]
            Name: str
            ExternalId: str
            DebateSections: list
                [65 items]
                    Title: str
                    ExternalId: str
                    ParentExternalId: NoneType
Success: bool
Errors: list
    [0 items]


In [5]:
TEST_START = "1945-01-01"
TEST_END   = "1950-12-31"


def search_spoken_contributions(
    term,
    start_date,
    end_date,
    house="Commons",
    skip=0,
    take=100,
    max_retries=3
):
    url = f"{API_BASE}/search/contributions/Spoken.json"

    params = {
        "queryParameters.house": house,
        "queryParameters.startDate": start_date,
        "queryParameters.endDate": end_date,
        "queryParameters.searchTerm": term,
        "queryParameters.skip": skip,
        "queryParameters.take": take
    }

    for attempt in range(max_retries):

        try:
            response = session.get(
                url,
                params=params,
                timeout=60
            )

            if response.status_code == 200:
                return response.json()

            print(
                f"{term} {start_date} to {end_date}: "
                f"HTTP {response.status_code}"
            )

            print(response.text[:300])

        except requests.RequestException as e:
            print(
                f"Request error "
                f"({attempt + 1}/{max_retries}):",
                e
            )

        time.sleep(2 ** attempt)

    return None

test_searches = {}

for word in TARGET_WORDS:

    test_searches[word] = (
        search_spoken_contributions(
            term=word,
            start_date=TEST_START,
            end_date=TEST_END,
            house="Commons",
            take=100
        )
    )

    time.sleep(1)

In [6]:
"""EUROPE_TERMS = [
    "Albania",
    "Armenia",
    "Austria",
    "Andorra",
    "Azerbaijan",
    "Belarus",
    "Belgium",
    "Bosnia and Herzegovina",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Denmark",
    "Czech Republic",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Georgia",
    "Hungary",
    "Ireland",
    "Italy",
    "Iceland",
    "Latvia",
    "Lithuania",
    "Luxembourg",
    "Liechtenstein",
    "Malta",
    "Moldova",
    "Monaco",
    "Montenegro",
    "Netherlands",
    "Norway",
    "Macedonia",
    "Poland",
    "Portugal",
    "Russia",
    "Soviet Union",
    "Romania",
    "Serbia",
    "Slovakia",
    "Slovenia",
    "Sweden",
    "Switzerland",
    "Spain",
    "San Marino",
    "Turkey",
    "Ukraine",
    "Yugoslavia"
]"""

EUROPE_TERMS = [
    "British Empire",
    "Empire",
    "Commonwealth",
    "Dominion",
    "Dominions",
    "Colony",
    "Colonies"
]


In [7]:
def get_term_count(
    term,
    start_date="1900-01-01",
    end_date="2000-12-31",
    house="Commons"
):
    result = search_spoken_contributions(
        term=term,
        start_date=start_date,
        end_date=end_date,
        house=house,
        take=1
    )

    if result is None:
        return None

    return result.get(
        "TotalResultCount"
    )


count_rows = []

for group_name, terms in [
    ("Europe / EU", EUROPE_TERMS),
]:

    for term in terms:

        for house in [
            "Commons",
            "Lords"
        ]:

            print(
                f"Searching: {term} / {house}"
            )

            count = get_term_count(
                term=term,
                house=house
            )

            count_rows.append({
                "group":
                    group_name,
                "term":
                    term,
                "house":
                    house,
                "count":
                    count
            })

            time.sleep(0.5)


term_counts = pd.DataFrame(
    count_rows
)

display(term_counts)

Searching: British Empire / Commons
Searching: British Empire / Lords
Searching: Empire / Commons
Searching: Empire / Lords
Searching: Commonwealth / Commons
Searching: Commonwealth / Lords
Searching: Dominion / Commons
Searching: Dominion / Lords
Searching: Dominions / Commons
Searching: Dominions / Lords
Searching: Colony / Commons
Searching: Colony / Lords
Searching: Colonies / Commons
Searching: Colonies / Lords


,group,term,house,count
0,Europe / EU,British Empire,Commons,15886
1,Europe / EU,British Empire,Lords,5034
2,Europe / EU,Empire,Commons,29819
3,Europe / EU,Empire,Lords,8815
4,Europe / EU,Commonwealth,Commons,48037
5,Europe / EU,Commonwealth,Lords,17386
6,Europe / EU,Dominion,Commons,9458
7,Europe / EU,Dominion,Lords,2123
8,Europe / EU,Dominions,Commons,14283
9,Europe / EU,Dominions,Lords,4340


In [9]:
FIVE_YEAR_TERMS = [
    "Empire",
    "Commonwealth"
]

five_year_rows = []

for start_year in range(1900, 2001, 5):
    end_year = min(start_year + 4, 2000)

    start_date = f"{start_year}-01-01"
    end_date = f"{end_year}-12-31"
    period = f"{start_year}-{end_year}"

    for term in FIVE_YEAR_TERMS:

        print(f"Searching: {term} / {period}")

        count = get_term_count(
            term=term,
            start_date=start_date,
            end_date=end_date,
            house="Commons"
        )

        five_year_rows.append({
            "period": period,
            "term": term,
            "count": count
        })

        time.sleep(0.5)

five_year_counts = pd.DataFrame(five_year_rows)

five_year_counts_wide = (
    five_year_counts
    .pivot(
        index="period",
        columns="term",
        values="count"
    )
    .reset_index()
)

display(five_year_counts_wide)

Searching: Empire / 1900-1904
Searching: Commonwealth / 1900-1904
Commonwealth 1900-01-01 to 1904-12-31: HTTP 500
{"Message":"An error has occurred."}
Searching: Empire / 1905-1909
Searching: Commonwealth / 1905-1909
Searching: Empire / 1910-1914
Searching: Commonwealth / 1910-1914
Searching: Empire / 1915-1919
Searching: Commonwealth / 1915-1919
Searching: Empire / 1920-1924
Searching: Commonwealth / 1920-1924
Searching: Empire / 1925-1929
Searching: Commonwealth / 1925-1929
Searching: Empire / 1930-1934
Searching: Commonwealth / 1930-1934
Searching: Empire / 1935-1939
Searching: Commonwealth / 1935-1939
Searching: Empire / 1940-1944
Searching: Commonwealth / 1940-1944
Searching: Empire / 1945-1949
Searching: Commonwealth / 1945-1949
Searching: Empire / 1950-1954
Searching: Commonwealth / 1950-1954
Searching: Empire / 1955-1959
Searching: Commonwealth / 1955-1959
Searching: Empire / 1960-1964
Searching: Commonwealth / 1960-1964
Searching: Empire / 1965-1969
Searching: Commonwealth / 1

term,period,Commonwealth,Empire
0,1900-1904,NaN,1706.0
1,1905-1909,152.0,1043.0
2,1910-1914,270.0,2121.0
3,1915-1919,150.0,1808.0
4,1920-1924,167.0,1885.0
5,1925-1929,280.0,2771.0
6,1930-1934,647.0,4572.0
7,1935-1939,591.0,3808.0
8,1940-1944,697.0,2223.0
9,1945-1949,2042.0,2344.0


In [17]:
term_counts_wide = (
    term_counts
    .pivot(
        index=[
            "group",
            "term"
        ],
        columns="house",
        values="count"
    )
    .reset_index()
)

if (
    "Commons" in term_counts_wide.columns
    and
    "Lords" in term_counts_wide.columns
):
    term_counts_wide[
        "Total"
    ] = (
        term_counts_wide["Commons"]
        .fillna(0)
        +
        term_counts_wide["Lords"]
        .fillna(0)
    )

term_counts_wide = (
    term_counts_wide
    .sort_values(
        [
            "group",
            "Total"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    term_counts_wide
)

house,group,term,Commons,Lords,Total
4,Europe / EU,Commonwealth,48037,17386,65423
2,Europe / EU,Colonies,35933,5814,41747
8,Europe / EU,Empire,29819,8815,38634
0,Europe / EU,British Commonwealth,15967,7408,23375
1,Europe / EU,British Empire,15886,5034,20920
7,Europe / EU,Dominions,14283,4340,18623
3,Europe / EU,Colony,9633,2676,12309
6,Europe / EU,Dominion,9458,2123,11581
5,Europe / EU,Commonwealth of Nations,7753,3782,11535


In [38]:
for word, data in test_searches.items():

    print("\n" + "=" * 70)
    print(word.upper())
    print("=" * 70)

    print(
        "Python type:",
        type(data).__name__
    )

    if isinstance(data, dict):

        print(
            "Top-level keys:",
            list(data.keys())
        )

        for key, value in data.items():

            if isinstance(value, list):
                print(
                    f"{key}: list with "
                    f"{len(value):,} items"
                )

            elif not isinstance(
                value,
                (dict, list)
            ):
                print(
                    f"{key}: {value}"
                )

    print("\nPreview:")
    print(
        json.dumps(
            data,
            indent=2
        )[:5000]
    )




WAR
Python type: dict
Top-level keys: ['SearchTerms', 'SpokenResultCount', 'WrittenResultCount', 'CorrectionsResultCount', 'DivisionsResultCount', 'Results', 'TotalResultCount']
SearchTerms: list with 1 items
SpokenResultCount: 0
WrittenResultCount: 0
CorrectionsResultCount: 0
DivisionsResultCount: 0
Results: list with 100 items
TotalResultCount: 30334

Preview:
{
  "SearchTerms": [
    "war"
  ],
  "SpokenResultCount": 0,
  "WrittenResultCount": 0,
  "CorrectionsResultCount": 0,
  "DivisionsResultCount": 0,
  "Results": [
    {
      "MemberName": "",
      "MemberId": -1,
      "AttributedTo": "Sir G. Jeffreys",
      "ItemId": 9080427,
      "ContributionExtId": "c95fec24-1a1d-466c-8474-bb8e0f7421ac",
      "ContributionText": "The best way to preserve peace is to be prepared for war.",
      "ContributionTextFull": "The best way to preserve peace is to be prepared for war.",
      "HRSTag": null,
      "HansardSection": null,
      "Timecode": null,
      "DebateSection": "Oil Ref

In [39]:
def find_result_list(data):

    preferred_keys = [
        "Results",
        "results",
        "Items",
        "items",
        "Contributions",
        "contributions"
    ]

    for key in preferred_keys:

        if (
            isinstance(data, dict)
            and key in data
            and isinstance(data[key], list)
        ):
            return key, data[key]

    if isinstance(data, dict):

        for key, value in data.items():

            if isinstance(value, list):
                return key, value

    return None, []


for word, data in test_searches.items():

    key, rows = find_result_list(
        data
    )

    print(
        f"\n{word}: "
        f"result key={key}, "
        f"rows={len(rows)}"
    )

    if rows:

        df = pd.json_normalize(
            rows
        )

        print(
            "Columns:"
        )

        print(
            df.columns.tolist()
        )

        display(
            df.head()
        )


war: result key=Results, rows=100
Columns:
['MemberName', 'MemberId', 'AttributedTo', 'ItemId', 'ContributionExtId', 'ContributionText', 'ContributionTextFull', 'HRSTag', 'HansardSection', 'Timecode', 'DebateSection', 'DebateSectionId', 'DebateSectionExtId', 'SittingDate', 'Section', 'House', 'OrderInDebateSection', 'DebateSectionOrder', 'Rank']


,MemberName,MemberId,AttributedTo,ItemId,ContributionExtId,ContributionText,ContributionTextFull,HRSTag,HansardSection,Timecode,DebateSection,DebateSectionId,DebateSectionExtId,SittingDate,Section,House,OrderInDebateSection,DebateSectionOrder,Rank
0,,-1.0,Sir G. Jeffreys,9080427,c95fec24-1a1d-466c-8474-bb8e0f7421ac,The best way to preserve peace is to be prepar...,The best way to preserve peace is to be prepar...,None,None,None,"Oil Refinery, The Solent",3069976,c0636361-82cc-45fd-8cb1-3100ab4cd8be,1950-12-15T00:00:00,Commons Chamber,Commons,40,10,96
1,,-1.0,Mr. Grimond (Orkney and Shetland),9080077,4740ad57-084e-4eaa-8b5d-8c82973c37ea,Among the many phrases which have been doing d...,Among the many phrases which have been doing d...,None,None,None,Raw Material Supplies,3069857,37aa6fa9-7fb3-418d-8165-b4192ee9e59e,1950-12-15T00:00:00,Commons Chamber,Commons,38,6,96
2,,-1.0,Mr. Chetwynd,9080091,c7035397-dafa-4274-8963-4ef58da90c95,"No, I think it would be far more just to say t...","No, I think it would be far more just to say t...",None,None,None,Raw Material Supplies,3069857,37aa6fa9-7fb3-418d-8165-b4192ee9e59e,1950-12-15T00:00:00,Commons Chamber,Commons,46,6,80
3,,-1.0,Sir G. Jeffreys,9080424,19156fa2-9f87-411f-91cf-93cd5f1aabc9,Is the hon. Gentleman aware that the port of S...,Is the hon. Gentleman aware that the port of S...,None,None,None,"Oil Refinery, The Solent",3069976,c0636361-82cc-45fd-8cb1-3100ab4cd8be,1950-12-15T00:00:00,Commons Chamber,Commons,36,10,64
4,,-1.0,Mr. Oliver Lyttelton (Aldershot),9080047,c763facd-d6af-49cc-970c-debe4548a5bc,Perhaps I can help the hon. Gentleman in the d...,Perhaps I can help the hon. Gentleman in the d...,None,None,None,Raw Material Supplies,3069857,37aa6fa9-7fb3-418d-8165-b4192ee9e59e,1950-12-15T00:00:00,Commons Chamber,Commons,18,6,64



peace: result key=Results, rows=100
Columns:
['MemberName', 'MemberId', 'AttributedTo', 'ItemId', 'ContributionExtId', 'ContributionText', 'ContributionTextFull', 'HRSTag', 'HansardSection', 'Timecode', 'DebateSection', 'DebateSectionId', 'DebateSectionExtId', 'SittingDate', 'Section', 'House', 'OrderInDebateSection', 'DebateSectionOrder', 'Rank']


,MemberName,MemberId,AttributedTo,ItemId,ContributionExtId,ContributionText,ContributionTextFull,HRSTag,HansardSection,Timecode,DebateSection,DebateSectionId,DebateSectionExtId,SittingDate,Section,House,OrderInDebateSection,DebateSectionOrder,Rank
0,,-1,Sir G. Jeffreys,9080427,c95fec24-1a1d-466c-8474-bb8e0f7421ac,The best way to preserve peace is to be prepar...,The best way to preserve peace is to be prepar...,None,None,None,"Oil Refinery, The Solent",3069976,c0636361-82cc-45fd-8cb1-3100ab4cd8be,1950-12-15T00:00:00,Commons Chamber,Commons,40,10,128
1,,-1,Mr. Lindgren,9080429,1f06d899-090b-4d5a-9602-15a6025af9a7,"I am not arguing on that basis, but if war com...","I am not arguing on that basis, but if war com...",None,None,None,"Oil Refinery, The Solent",3069976,c0636361-82cc-45fd-8cb1-3100ab4cd8be,1950-12-15T00:00:00,Commons Chamber,Commons,42,10,85
2,,-1,Mr. Lindgren,9080426,a1a8bd59-6f13-4ac3-81d7-5aee8472547e,The hon. and gallant Gentleman must remember t...,The hon. and gallant Gentleman must remember t...,None,None,None,"Oil Refinery, The Solent",3069976,c0636361-82cc-45fd-8cb1-3100ab4cd8be,1950-12-15T00:00:00,Commons Chamber,Commons,38,10,64
3,,-1,Mr. Grimond (Orkney and Shetland),9080077,4740ad57-084e-4eaa-8b5d-8c82973c37ea,Among the many phrases which have been doing d...,Among the many phrases which have been doing d...,None,None,None,Raw Material Supplies,3069857,37aa6fa9-7fb3-418d-8165-b4192ee9e59e,1950-12-15T00:00:00,Commons Chamber,Commons,38,6,48
4,,-1,Mr. Oliver Lyttelton (Aldershot),9080047,c763facd-d6af-49cc-970c-debe4548a5bc,Perhaps I can help the hon. Gentleman in the d...,Perhaps I can help the hon. Gentleman in the d...,None,None,None,Raw Material Supplies,3069857,37aa6fa9-7fb3-418d-8165-b4192ee9e59e,1950-12-15T00:00:00,Commons Chamber,Commons,18,6,42



security: result key=Results, rows=100
Columns:
['MemberName', 'MemberId', 'AttributedTo', 'ItemId', 'ContributionExtId', 'ContributionText', 'ContributionTextFull', 'HRSTag', 'HansardSection', 'Timecode', 'DebateSection', 'DebateSectionId', 'DebateSectionExtId', 'SittingDate', 'Section', 'House', 'OrderInDebateSection', 'DebateSectionOrder', 'Rank']


,MemberName,MemberId,AttributedTo,ItemId,ContributionExtId,ContributionText,ContributionTextFull,HRSTag,HansardSection,Timecode,DebateSection,DebateSectionId,DebateSectionExtId,SittingDate,Section,House,OrderInDebateSection,DebateSectionOrder,Rank
0,,-1,Mr. Oliver Lyttelton (Aldershot),9080417,4983ea2f-6fc8-4f25-9782-085ff79492d7,I entirely agree with many of the remarks made...,I entirely agree with many of the remarks made...,None,None,None,"Oil Refinery, The Solent",3069976,c0636361-82cc-45fd-8cb1-3100ab4cd8be,1950-12-15T00:00:00,Commons Chamber,Commons,28,10,38
1,,-1,Mr. Paton,9079830,5a315bc3-9e82-440e-856a-9d1bfb4d5882,"I am sorry, but I cannot give way. There is on...","I am sorry, but I cannot give way.<p></p> Ther...",None,None,None,Prime Minister's Visit To Usa,3069686,1f8b6bd6-f5fd-423b-9316-fa46bfd30969,1950-12-14T00:00:00,Commons Chamber,Commons,252,48,64
2,,-1,Mr. Davies,9079090,d064a1dc-8cd3-4e58-99be-c5ba581b5721,"Does the Home Secretary suggest that, tiresome...","Does the Home Secretary suggest that, tiresome...",None,None,None,Interrogation,3069576,b0e69302-7896-4472-ab55-225293e12fab,1950-12-14T00:00:00,Commons Chamber,Commons,11,27,48
3,,-1,Mr. Molson (The High Peak),9079621,5d783a3e-995f-4fe8-b76c-a5566d57f4c9,May I ask the right hon. and learned Gentleman...,May I ask the right hon. and learned Gentleman...,None,None,None,Prime Minister's Visit To Usa,3069686,1f8b6bd6-f5fd-423b-9316-fa46bfd30969,1950-12-14T00:00:00,Commons Chamber,Commons,114,48,32
4,,-1,Mr. Janner,9079432,1ea54598-a4de-49cc-843e-3b8bf211a35a,In view of the very considerable anxiety that ...,In view of the very considerable anxiety that ...,None,None,None,Business Of The House,3069671,bc06f779-b97d-44f3-84aa-68ba5f13978b,1950-12-14T00:00:00,Commons Chamber,Commons,20,43,32


In [40]:
house_tests = {}

for house in [
    "Commons",
    "Lords"
]:

    house_tests[house] = {}

    for word in TARGET_WORDS:

        house_tests[house][word] = (
            search_spoken_contributions(
                term=word,
                start_date=TEST_START,
                end_date=TEST_END,
                house=house,
                take=20
            )
        )

        time.sleep(1)

Attempt 1: 500
{"Message":"An error has occurred."}


In [41]:
def get_historic_sitting_days(
    start_date,
    end_date,
    house="Commons",
    skip=0,
    take=100
):

    url = (
        f"{API_BASE}/"
        "historicsittingdays"
    )

    params = {
        "queryParams.house":
            house,
        "queryParams.startDate":
            start_date,
        "queryParams.endDate":
            end_date,
        "queryParams.skip":
            skip,
        "queryParams.take":
            take,
        "queryParams.orderBy":
            "SittingDateAsc",
        "queryParams.hasSittingSections":
            True
    }

    response = session.get(
        url,
        params=params,
        timeout=60
    )

    print(
        "Status:",
        response.status_code
    )

    print(
        "URL:",
        response.url
    )

    if response.status_code != 200:
        print(
            response.text[:3000]
        )

    response.raise_for_status()

    return response.json()


sitting_test = (
    get_historic_sitting_days(
        "1949-05-01",
        "1949-05-07",
        house="Commons"
    )
)

print(
    json.dumps(
        sitting_test,
        indent=2
    )[:10000]
)

Status: 200
URL: https://hansard-api.parliament.uk/historicsittingdays?queryParams.house=Commons&queryParams.startDate=1949-05-01&queryParams.endDate=1949-05-07&queryParams.skip=0&queryParams.take=100&queryParams.orderBy=SittingDateAsc&queryParams.hasSittingSections=True
{
  "Item": {
    "Results": [
      {
        "HouseId": 1,
        "SittingDate": "1949-05-02T00:00:00"
      },
      {
        "HouseId": 1,
        "SittingDate": "1949-05-03T00:00:00"
      },
      {
        "HouseId": 1,
        "SittingDate": "1949-05-04T00:00:00"
      },
      {
        "HouseId": 1,
        "SittingDate": "1949-05-05T00:00:00"
      },
      {
        "HouseId": 1,
        "SittingDate": "1949-05-06T00:00:00"
      }
    ],
    "TotalResultCount": 5
  },
  "Success": true,
  "Errors": []
}


In [42]:
def search_member(
    name,
    date=None,
    house="Commons"
):
    url = f"{API_BASE}/search/members.json"

    params = {
        "queryParameters.searchTerm": name,
        "queryParameters.house": house,
        "queryParameters.includeFormer": True,
        "queryParameters.includeCurrent": True,
        "queryParameters.skip": 0,
        "queryParameters.take": 20,
    }

    if date is not None:
        params["queryParameters.date"] = date

    response = session.get(
        url,
        params=params,
        timeout=30
    )

    print("Status:", response.status_code)
    print("URL:", response.url)

    if response.status_code != 200:
        print(response.text[:3000])

    response.raise_for_status()

    return response.json()

member_test = search_member(
    name="Sir J. Grigg",
    date="1945-01-16",
    house="Commons"
)


print(
    json.dumps(
        member_test,
        indent=2
    )[:10000]
)

def find_member_rows(data):
    if not isinstance(data, dict):
        return []

    for key in [
        "Members",
        "Results",
        "Items"
    ]:
        if (
            key in data
            and isinstance(data[key], list)
        ):
            return data[key]

    for value in data.values():
        if isinstance(value, list):
            return value

    return []


member_rows = find_member_rows(
    member_test
)

print(
    "Member rows:",
    len(member_rows)
)

if member_rows:

    member_df = pd.json_normalize(
        member_rows
    )

    print(member_df.columns.tolist())

    display(member_df)

Status: 200
URL: https://hansard-api.parliament.uk/search/members.json?queryParameters.searchTerm=Sir+J.+Grigg&queryParameters.house=Commons&queryParameters.includeFormer=True&queryParameters.includeCurrent=True&queryParameters.skip=0&queryParameters.take=20&queryParameters.date=1945-01-16
{
  "Results": [],
  "TotalResultCount": 0
}
Member rows: 0


In [43]:
security_df = pd.json_normalize(
    test_searches["security"]["Results"]
)

display(
    security_df[
        [
            "AttributedTo",
            "MemberId",
            "SittingDate",
            "DebateSection",
            "ContributionTextFull"
        ]
    ].head(10)
)

,AttributedTo,MemberId,SittingDate,DebateSection,ContributionTextFull
0,Mr. Oliver Lyttelton (Aldershot),-1,1950-12-15T00:00:00,"Oil Refinery, The Solent",I entirely agree with many of the remarks made...
1,Mr. Paton,-1,1950-12-14T00:00:00,Prime Minister's Visit To Usa,"I am sorry, but I cannot give way.<p></p> Ther..."
2,Mr. Davies,-1,1950-12-14T00:00:00,Interrogation,"Does the Home Secretary suggest that, tiresome..."
3,Mr. Molson (The High Peak),-1,1950-12-14T00:00:00,Prime Minister's Visit To Usa,May I ask the right hon. and learned Gentleman...
4,Mr. Janner,-1,1950-12-14T00:00:00,Business Of The House,In view of the very considerable anxiety that ...
5,Mr. Ede,-1,1950-12-14T00:00:00,Soviet Trade Union Delegation (Visas),"Before these visas were granted, I took the pr..."
6,Mr. Ede,-1,1950-12-14T00:00:00,Interrogation,In many foreign countries which have a land fr...
7,Mr. Eden,-1,1950-12-14T00:00:00,Prime Minister's Visit To Usa,"All right, but if we have a finger in every pi..."
8,"Mr. Julian Amery (Preston, North)",-1,1950-12-14T00:00:00,Prime Minister's Visit To Usa,The hon. Member for Leek (Mr. Harold Davies) a...
9,Mr. Walker-Smith,-1,1950-12-14T00:00:00,Prime Minister's Visit To Usa,"I hate to refuse to give way, but there is not..."


In [44]:
speaker_text = (
    security_df
    .iloc[0]["AttributedTo"]
)

speaker_date = (
    security_df
    .iloc[0]["SittingDate"][:10]
)

print(
    "Speaker:",
    speaker_text
)

print(
    "Date:",
    speaker_date
)

Speaker: Mr. Oliver Lyttelton  (Aldershot)
Date: 1950-12-15


In [ ]:
later_test = search_spoken_contributions(
    term="security",
    start_date="1975-01-01",
    end_date="1975-01-31",
    house="Commons",
    take=50
)

if later_test is not None:

    later_df = pd.json_normalize(
        later_test.get(
            "Results",
            []
        )
    )

    if len(later_df) > 0:

        display(
            later_df[
                [
                    "MemberName",
                    "MemberId",
                    "AttributedTo",
                    "SittingDate",
                    "DebateSection"
                ]
            ].head(20)
        )

    else:
        print("No results returned.")

else:
    print("API request failed.")

display(
    later_df[
        [
            "MemberName",
            "MemberId",
            "AttributedTo",
            "SittingDate",
            "DebateSection"
        ]
    ].head(20)
)

Attempt 1: 500
{"Message":"An error has occurred."}


KeyError: 'Results'

In [ ]:
print(
    "Rows:",
    len(later_df)
)

print(
    "Valid MemberId:",
    (
        later_df["MemberId"]
        .fillna(-1)
        .ne(-1)
        .sum()
    )
)

print(
    "Valid MemberId %:",
    round(
        (
            later_df["MemberId"]
            .fillna(-1)
            .ne(-1)
            .mean()
            * 100
        ),
        1
    )
)

Rows: 100
Valid MemberId: 0
Valid MemberId %: 0.0


In [ ]:
TEST_PERIODS = [
    ("1905", "1905-01-01", "1905-01-31"),
    ("1925", "1925-01-01", "1925-01-31"),
    ("1945", "1945-01-01", "1945-01-31"),
    ("1965", "1965-01-01", "1965-01-31"),
    ("1985", "1985-01-01", "1985-01-31"),
    ("2000", "2000-01-01", "2000-01-31"),
]

audit_rows = []

for label, start_date, end_date in TEST_PERIODS:

    result = search_spoken_contributions(
        term="security",
        start_date=start_date,
        end_date=end_date,
        house="Commons",
        take=50
    )

    if result is None:
        audit_rows.append({
            "year": label,
            "status": "failed"
        })
        continue

    rows = result.get("Results", [])

    if not rows:
        audit_rows.append({
            "year": label,
            "status": "no results"
        })
        continue

    df = pd.json_normalize(rows)

    audit_rows.append({
        "year": label,
        "status": "ok",
        "rows_returned": len(df),
        "total_results":
            result.get("TotalResultCount"),
        "valid_member_id_pct":
            df["MemberId"]
            .fillna(-1)
            .ne(-1)
            .mean() * 100,
        "speaker_label_pct":
            df["AttributedTo"]
            .fillna("")
            .str.strip()
            .ne("")
            .mean() * 100,
        "debate_title_pct":
            df["DebateSection"]
            .fillna("")
            .str.strip()
            .ne("")
            .mean() * 100
    })

    time.sleep(1)

audit_df = pd.DataFrame(audit_rows)

display(audit_df)

security / Commons -> 200
security / Commons -> 200
security / Commons -> 500
{"Message":"An error has occurred."}


HTTPError: 500 Server Error: Internal Server Error for url: https://hansard-api.parliament.uk/search/contributions/Spoken.json?queryParameters.house=Commons&queryParameters.startDate=1945-01-01&queryParameters.endDate=1945-12-31&queryParameters.searchTerm=security&queryParameters.skip=0&queryParameters.take=100&queryParameters.orderBy=SittingDateAsc

In [ ]:
def test_member_names(
    names,
    date=None,
    house="Commons"
):
    results = {}

    for name in names:

        data = search_member(
            name=name,
            date=date,
            house=house
        )

        results[name] = data

        print(
            "\n",
            "=" * 60,
            "\n",
            name
        )

        print(
            json.dumps(
                data,
                indent=2
            )[:3000]
        )

        time.sleep(0.5)

    return results


member_name_tests = test_member_names(
    [
        "Grigg",
        "Strauss",
        "Bevin"
    ],
    date="1945-01-16",
    house="Commons"
)

In [ ]:
TEST_DEBATE_ID = (
    test_searches[
        "security"
    ]["Results"][0][
        "DebateSectionExtId"
    ]
)

url = (
    f"{API_BASE}/debates/"
    f"speakerslist/"
    f"{TEST_DEBATE_ID}.json"
)

response = session.get(
    url,
    timeout=30
)

print(
    "Status:",
    response.status_code
)

print(
    response.text[:5000]
)